# PaddleOCR v4 Inference (Google Colab)
This notebook downloads the AAR rulings image dataset from Hugging Face and runs PaddleOCR v4 to extract text, benchmarking the performance.

In [ ]:
# Install PaddlePaddle GPU version (suitable for Colab's default T4 GPU)
!python -m pip install paddlepaddle-gpu -i https://pypi.tuna.tsinghua.edu.cn/simple

# Install PaddleOCR and dataset handling libraries
!pip install paddleocr datasets pandas tqdm pillow numpy

In [ ]:
import time
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
from paddleocr import PaddleOCR

In [ ]:
# Load dataset from Hugging Face
# Replace 'your_hf_username' with your actual username/dataset location
dataset_name = "your_hf_username/aar_rulings_ocr_sample"
print(f"Loading dataset: {dataset_name}")
try:
    dataset = load_dataset(dataset_name, split="train")
    print(f"Loaded {len(dataset)} images.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Make sure the dataset is public, or that you've logged in with your HF Token.")

In [ ]:
# Initialize PaddleOCR with PP-OCRv4 (mobile_det by default for v4 usually, or you can specify models if needed)
# It will automatically download the required model weights on the first run.
ocr = PaddleOCR(use_angle_cls=True, lang='en', ocr_version='PP-OCRv4')

In [ ]:
# Run inference
results = []
start_time_total = time.time()

if 'dataset' in locals():
    for idx, item in enumerate(tqdm(dataset, desc="Running PaddleOCR v4")):
        img = item["image"]
        pdf_name = item["pdf_name"]
        page_num = item["page_num"]
        
        # PaddleOCR expects numpy array (BGR format is typical for cv2, but RGB is fine usually)
        # The PIL image from Hugging Face is RGB
        img_np = np.array(img)
        
        start_time = time.time()
        error = None
        text = ""
        try:
            # Result is a list of lists containing bounding boxes and text
            result = ocr.ocr(img_np, cls=True)
            # PaddleOCR returns None if no text is detected on page
            if result and result[0]:
                text_parts = [line[1][0] for line in result[0]]
                text = "\n".join(text_parts)
        except Exception as e:
            error = str(e)
            
        runtime = time.time() - start_time
        
        results.append({
            "pdf_name": pdf_name,
            "page_num": page_num,
            "runtime_seconds": runtime,
            "extracted_text_length": len(text),
            "error": error
        })

    total_runtime = time.time() - start_time_total
    df_results = pd.DataFrame(results)
else:
    print("Dataset not loaded, skipping inference.")

In [ ]:
# Metrics & Saving
if 'df_results' in locals() and not df_results.empty:
    avg_speed = df_results["runtime_seconds"].mean()
    errors_count = df_results["error"].notnull().sum()

    print(f"Total Runtime: {total_runtime:.2f} seconds")
    print(f"Average Speed: {avg_speed:.4f} seconds/page")
    print(f"Total Errors: {errors_count}")

    output_file = "paddleocr_results.csv"
    df_results.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")
